In [1]:
import numpy as np

In [ ]:
# TODO las interacciones normativas son más comunes que las empíricas

In [38]:
PHI = 1 # dinero
Q   = 1000 # cantidad de interacciones normativas
N   = 1000 # cantidad de jugadores

In [108]:
class Jugador:
    def __init__(self, dinero: int = PHI,W_nor_pos=0.1,W_nor_neg=0.1):
        self.dinero = dinero
        self.dictador           = False
        self.aspiracion_alfa    = np.random.uniform(0, dinero) # aspiración inicial
        self.suscept_norm       = 0 # susceptibilidad a estímulos normativos (dictador)
        # self.suscept_emp        = 1-self.suscept_norm     # susceptibilidad a estímulos empíricos (receptor)
        self.ruido_delta_norm        = np.random.uniform(0, dinero) # ruido en la actualización de la aspiración
        
        # Pesos para estímulos normativos positivos y negativos
        self.W_nor_pos = W_nor_pos  # peso para estímulos normativos positivos
        self.W_nor_neg = W_nor_neg  # peso para estímulos normativos negativos

        self.dinero_donado_theta: float = max(0,min(dinero,self.aspiracion_alfa*(1+self.ruido_delta_norm)))
        
    def convertir_en_dictador(self):
        self.dictador = True
        
    def interaccion_normativa(self,dictadores:list,interacciones=Q):
        "generamos interacción normativa entre dictadores y devolvemos la actualización de la aspiración del receptor"
        self.suscept_norm = sum([dictad.dinero_donado_theta for dictad in dictadores])/self.dinero*interacciones - self.aspiracion_alfa/self.dinero

    def actualizar_aspiracion(self):
        """
        Actualizamos la aspiración del receptor según la interacción normativa
        Basado en la fórmula (5): I^nor_{j,t} = I^nor_{j,t-1} * (1 + W^{nor,pos/neg} * ξ^nor_{j,t})
        
        - Si ξ^nor_{j,t} ≥ 0: usamos W^{nor,pos} (estímulo normativo positivo)
        - Si ξ^nor_{j,t} < 0: usamos W^{nor,neg} (estímulo normativo negativo)
        """
        self.interaccion_normativa()
        
        # Determinamos si el estímulo es positivo o negativo
        if self.suscept_norm >= 0:
            # Estímulo positivo: la aspiración tiende a aumentar
            self.suscept_norm*= (1 + self.W_nor_pos)

            self.aspiracion_alfa+=(self.dinero-self.aspiracion_alfa)*self.suscept_norm
        else:
            # Estímulo negativo: la aspiración tiende a disminuir
            self.suscept_norm*= (1 + self.W_nor_neg)

            self.aspiracion_alfa+=(self.dinero-self.aspiracion_alfa)*self.suscept_norm


In [103]:
jugadores = [Jugador() for i in range(N)]
for jugador in jugadores:
    jugador.convertir_en_dictador() if np.random.rand() < 0.5 else None

In [104]:
jugadores[0].interaccion_normativa([jug for jug in jugadores if jug.dictador and jug != jugadores[0]],dinero=PHI,interacciones=Q)

In [105]:
jugadores[0].dinero_donado_theta

0.8387001207192034